# TI-748: Causal Impact Analysis — Media Plan Feature (v5)

---

## Glossary & Key Definitions

| Term | Definition |
|---|---|
| **Media Plan** | A beta feature giving advertisers control over network allocation (% of spend per publisher like ABC, CBS, ESPN). MNTN auto-recommends allocations; advertisers can accept or customize. |
| **Recommended** | An advertiser who accepted MNTN's suggested network allocation without modification (`badge_state = 'RECOMMENDED'` on all publishers). |
| **Customized** | An advertiser who modified the allocation (`badge_state = 'USER_MODIFIED'` or `'USER_ADDED'`). |
| **CausalImpact** | Google's Bayesian Structural Time Series method for estimating the causal effect of an intervention on a time series. |
| **Intervention Date** | The date each advertiser first created a recommended media plan (`media_plan.create_time`). |
| **Pre-period** | The time before the intervention — used to train the model on the advertiser's baseline behavior. We use 52 weeks (1 full year) to capture seasonality. |
| **Post-period** | The time after the intervention (excluding the first 4 ramp-up weeks) — the model compares actual performance to what it predicted would have happened without the intervention. |
| **Ramp-up period** | The first 4 weeks after intervention, excluded from analysis. New campaigns need time for bidder learning, frequency buildup, and delivery footprint exploration before reaching steady-state performance (TI-780, N=6,917 campaigns). |
| **Counterfactual** | The model's prediction of what *would have happened* if the advertiser had NOT adopted media plan. |
| **Relative Effect** | The % difference between actual performance and the counterfactual: `(actual - predicted) / predicted`. Positive = improvement. |
| **p-value** | The probability of seeing this effect (or more extreme) if the intervention had no real impact. p < 0.05 = statistically significant. |
| **Spend-Weighted Average** | An average where each advertiser's effect is weighted by their total post-period spend. This prevents small advertisers (with noisy data) from disproportionately influencing the overall result. |
| **Covariates** | External variables the model uses to build a better counterfactual. Selected per-advertiser using BIC (not hand-picked). |
| **VIF (Variance Inflation Factor)** | Detects multicollinearity between covariates. VIF > 10 means >90% of a covariate's variance is explained by other covariates — coefficient estimates become unstable. We iteratively remove the worst offenders. |
| **BIC (Bayesian Information Criterion)** | Model selection criterion that balances fit vs complexity. Lower = better. BIC penalizes extra parameters more heavily than AIC, producing more parsimonious models that generalize better with small N. |
| **Panel Data Model** | A pooled regression across all advertisers with advertiser fixed effects and time controls. Produces one aggregate treatment effect estimate with proper standard errors. Complementary to per-advertiser CausalImpact. |
| **Placebo Test** | A validation test: we pretend the intervention happened at multiple points in the pre-period. If the model is reliable, it should find NO significant effect (because nothing actually changed). |
| **Winsorize** | Clipping extreme values to the 1st/99th percentile to reduce the influence of data anomalies. |

### Metrics

| Metric | Formula | Better When | What It Measures |
|---|---|---|---|
| **IVR** | verified_visits / impressions | Higher | How often an ad impression leads to a site visit |
| **CVR** | conversions / verified_visits | Higher | How often a visit leads to a conversion |
| **CPA** | spend / conversions | Lower | Cost to acquire one conversion |
| **CPV** | spend / verified_visits | Lower | Cost to generate one site visit |
| **ROAS** | order_value / spend | Higher | Revenue generated per dollar spent |

---

## Executive Summary

**Objective:** Determine if advertisers who adopted MNTN's *recommended* Media Plan network allocation saw improved prospecting performance.

**Two analyses:**
1. **CausalImpact (time-series):** For each adopter, compare their advertiser-level prospecting performance before vs after adopting media plan, controlling for market trends via BIC-selected covariates per advertiser.
2. **Panel data model:** Pooled regression across all advertisers with fixed effects — one aggregate treatment estimate.
3. **Within-advertiser comparison (cross-sectional):** For advertisers with both recommended and non-recommended campaigns, compare performance side-by-side (after 4-week ramp-up exclusion).

| Design Element | Choice | Rationale |
|---|---|---|
| Unit of analysis | Advertiser (all prospecting campaigns) | Feature affects portfolio-level network allocation |
| Intervention date | First recommended plan `create_time` | When they actually started using the feature |
| Pre-period | 52 weeks (max available) | Full seasonal cycle — captures Black Friday, holidays, Q1 slowdown |
| Post-period | Variable (5-18 weeks) | Depends on adoption date; first 4 weeks excluded (ramp-up) |
| Ramp-up exclusion | 4 weeks post-intervention | TI-780: new campaigns reach 89% of steady-state IVR by week 4 (N=6,917) |
| Covariate selection | BIC per-advertiser (from 14 candidates) | Data-driven selection avoids hand-picking bias; BIC penalizes complexity |
| Multicollinearity | VIF iterative elimination (threshold > 10) | Removes redundant covariates before BIC selection |
| Filter | Recommended plans only | Per Kirsa: only care about advertisers who used MNTN's recommended allocation |
| Validation | Multi-point placebo tests (5 per advertiser), sensitivity analysis (5 pre-period lengths) | 24% placebo FPR; 5/6 directionally consistent |

---

## 1. Setup & Configuration

In [ ]:
# !pip install google-cloud-bigquery pycausalimpact pandas numpy matplotlib statsmodels db-dtypes

import warnings
from datetime import date
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from causalimpact import CausalImpact
from google.cloud import bigquery

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*frequency information.*")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

BQ_PROJECT = "dw-main-silver"
MIN_WEEKLY_IMPRESSIONS = 1000
MIN_TOTAL_POST_SPEND = 10_000
MIN_PRE_WEEKS = 20
MIN_POST_WEEKS = 4
PRIMARY_METRIC = "ivr"
RAMP_UP_WEEKS = 4  # TI-780: campaigns reach 89% of steady-state IVR by week 4

# Candidate covariates for BIC selection (per advertiser).
# BIC will pick the best subset — we don't hand-pick.
COVARIATE_CANDIDATES = [
    "platform_ivr", "platform_spend", "platform_impressions",
    "platform_active_advertisers", "platform_vcr",
    "holiday",
    "metric_lag1", "metric_lag2",
    "spend_change_pct",
]

METRIC_DEFS = {
    "ivr":  {"direction": "higher", "label": "Impression-to-Visit Rate"},
    "cvr":  {"direction": "higher", "label": "Conversion Rate"},
    "cpa":  {"direction": "lower",  "label": "Cost per Acquisition"},
    "cpv":  {"direction": "lower",  "label": "Cost per Visit"},
    "roas": {"direction": "higher", "label": "Return on Ad Spend"},
}

HOLIDAY_WEEKS = {
    pd.Timestamp("2024-11-25"): 1, pd.Timestamp("2024-12-23"): 1, pd.Timestamp("2024-12-30"): 1,
    pd.Timestamp("2025-11-24"): 1, pd.Timestamp("2025-12-22"): 1, pd.Timestamp("2025-12-29"): 1,
    pd.Timestamp("2026-02-02"): 1,
}

client = bigquery.Client(project=BQ_PROJECT)
print("Setup complete.")

## 2. Identify Recommended Media Plan Adopters

Only advertisers with `badge_state = 'RECOMMENDED'` on ALL publishers in at least one active plan.

In [ ]:
adopters = client.query("""
WITH plan_status AS (
    SELECT mp.media_plan_id, mp.advertiser_id, mp.campaign_group_id, mp.create_time,
           LOGICAL_AND(mpp.badge_state = 'RECOMMENDED') AS all_recommended
    FROM `dw-main-silver.core.media_plan` mp
    JOIN `dw-main-silver.core.media_plan_publishers` mpp ON mpp.media_plan_id = mp.media_plan_id
    WHERE mp.media_plan_status_id = 3
    GROUP BY 1, 2, 3, 4
)
SELECT ps.advertiser_id, adv.company_name,
    MIN(CASE WHEN ps.all_recommended THEN ps.create_time END) AS first_recommended_plan,
    COUNT(*) AS total_plans,
    COUNTIF(ps.all_recommended) AS recommended_plans,
    COUNTIF(NOT ps.all_recommended) AS customized_plans
FROM plan_status ps
JOIN `dw-main-bronze.integrationprod.advertisers` adv ON adv.advertiser_id = ps.advertiser_id
GROUP BY 1, 2
HAVING COUNTIF(ps.all_recommended) > 0
ORDER BY 3
""").to_dataframe()

adopters["first_recommended_plan"] = pd.to_datetime(adopters["first_recommended_plan"])
adopters["intervention_date"] = adopters["first_recommended_plan"].dt.date

print(f"{len(adopters)} advertisers with recommended media plans")
adopters[["advertiser_id", "company_name", "intervention_date", "recommended_plans", "customized_plans"]]

## 3. Load Weekly Prospecting KPIs

- **Source:** `summarydata.sum_by_campaign_by_day` (2024-01-01 to present — 15+ months of history)
- **Filter:** Prospecting campaigns only (`funnel_level = 1`), non-test, non-deleted
- **Attribution:** All adopters use `industry_standard` (competing views included)
- **Quality:** Weeks with < 1,000 impressions excluded (attribution lag artifacts)

In [ ]:
adopter_list = ",".join(str(x) for x in adopters["advertiser_id"])

weekly_kpis = client.query(f"""
WITH prospecting AS (
    SELECT DISTINCT c.campaign_id, c.advertiser_id, c.campaign_group_id
    FROM `dw-main-bronze.integrationprod.campaigns` c
    WHERE c.funnel_level = 1 AND c.deleted = FALSE AND c.is_test = FALSE
)
SELECT pc.advertiser_id, pc.campaign_group_id,
    DATE_TRUNC(s.day, WEEK(MONDAY)) AS week_start,
    CASE WHEN pc.advertiser_id IN ({adopter_list}) THEN TRUE ELSE FALSE END AS is_adopter,
    SUM(s.impressions) AS impressions,
    SUM(s.media_spend + s.data_spend + s.platform_spend) AS spend,
    SUM(s.clicks + s.views + COALESCE(s.competing_views, 0)) AS vv,
    SUM(s.click_conversions + s.view_conversions + COALESCE(s.competing_view_conversions, 0)) AS conversions,
    SUM(s.click_order_value + s.view_order_value + COALESCE(s.competing_view_order_value, 0)) AS order_value,
    SUM(s.vast_start) AS vast_start, SUM(s.vast_complete) AS vast_complete
FROM `dw-main-silver.summarydata.sum_by_campaign_by_day` s
JOIN prospecting pc ON pc.campaign_id = s.campaign_id
WHERE s.day >= '2024-01-01' AND s.impressions > 0
GROUP BY 1, 2, 3, 4 ORDER BY 1, 3
""").to_dataframe()

weekly_kpis["week_start"] = pd.to_datetime(weekly_kpis["week_start"])
for c in ["impressions", "spend", "vv", "conversions", "order_value", "vast_start", "vast_complete"]:
    weekly_kpis[c] = pd.to_numeric(weekly_kpis[c], errors="coerce").astype(float)
weekly_kpis = weekly_kpis[weekly_kpis["impressions"] >= MIN_WEEKLY_IMPRESSIONS].copy()

print(f"{len(weekly_kpis):,} records | {weekly_kpis['advertiser_id'].nunique()} advertisers")
print(f"Date range: {weekly_kpis['week_start'].min().date()} to {weekly_kpis['week_start'].max().date()}")

## 4. Compute Metrics & Platform Covariates

Advertiser-specific covariates (`spend_change_pct`, `metric_lag1/2`) are added per-advertiser in the analysis step.

In [ ]:
def compute_metrics(df):
    df = df.copy()
    df["ivr"] = df["vv"] / df["impressions"].replace(0, np.nan)
    df["cvr"] = df["conversions"] / df["vv"].replace(0, np.nan)
    df["cpa"] = df["spend"] / df["conversions"].replace(0, np.nan)
    df["cpv"] = df["spend"] / df["vv"].replace(0, np.nan)
    df["roas"] = df["order_value"] / df["spend"].replace(0, np.nan)
    df["vcr"] = df["vast_complete"] / df["vast_start"].replace(0, np.nan)
    return df

def winsorize(s, lo=1, hi=99):
    return s.clip(lower=np.nanpercentile(s.dropna(), lo), upper=np.nanpercentile(s.dropna(), hi))

# Aggregate to advertiser-week
adv_week = weekly_kpis.groupby(["advertiser_id", "week_start", "is_adopter"]).agg(
    impressions=("impressions", "sum"), spend=("spend", "sum"), vv=("vv", "sum"),
    conversions=("conversions", "sum"), order_value=("order_value", "sum"),
    vast_start=("vast_start", "sum"), vast_complete=("vast_complete", "sum"),
).reset_index()

# Platform covariates from non-adopters
non_adopter = adv_week[~adv_week["is_adopter"]]
platform = non_adopter.groupby("week_start").agg(
    platform_impressions=("impressions", "sum"), platform_spend=("spend", "sum"),
    platform_vv=("vv", "sum"), platform_vast_start=("vast_start", "sum"),
    platform_vast_complete=("vast_complete", "sum"),
    platform_active_advertisers=("advertiser_id", "nunique"),
).reset_index()
platform["platform_ivr"] = platform["platform_vv"] / platform["platform_impressions"].replace(0, np.nan)
platform["platform_vcr"] = platform["platform_vast_complete"] / platform["platform_vast_start"].replace(0, np.nan)
platform["holiday"] = platform["week_start"].map(lambda w: HOLIDAY_WEEKS.get(w, 0.0))
platform["platform_spend"] /= 1e6
platform["platform_impressions"] /= 1e9
platform["platform_active_advertisers"] /= 1000.0

print(f"Platform covariates: {len(platform)} weeks from {non_adopter['advertiser_id'].nunique()} non-adopter advertisers")
platform.tail()

## 5. BIC Covariate Selection & CausalImpact Analysis

**v5 methodology:**
1. Start with 14 candidate covariates (platform metrics, holidays, lags, spend changes)
2. For each advertiser, test all covariate subsets (size 1-6) using BIC on the pre-period
3. Select the lowest-BIC subset — data-driven, per-advertiser, no hand-picking
4. Exclude first 4 weeks post-intervention (ramp-up period, TI-780)
5. Run CausalImpact with the BIC-selected covariates

In [ ]:
def select_covariates_bic(adv_data, metric, candidates, pre_period):
    """
    BIC stepwise covariate selection for a single advertiser.
    Tests all subsets of size 1-6, returns the set with lowest BIC.
    """
    pre_data = adv_data.loc[pre_period[0]:pre_period[1]]
    y = pre_data[metric].dropna()
    available = [c for c in candidates if c in adv_data.columns]

    best_bic = np.inf
    best_covs = available[:2]  # fallback

    for size in range(1, min(len(available) + 1, 7)):
        for combo in combinations(available, size):
            combo_list = list(combo)
            X = pre_data[combo_list].reindex(y.index).dropna()
            common = y.index.intersection(X.index)
            if len(common) < 20:
                continue
            try:
                model = sm.OLS(y.loc[common], sm.add_constant(X.loc[common])).fit()
                if model.bic < best_bic:
                    best_bic = model.bic
                    best_covs = combo_list
            except Exception:
                pass

    return best_covs


def run_ci(adv_id, intervention_date, metric, adv_week, platform):
    """Run CausalImpact for one advertiser with BIC covariate selection and ramp-up exclusion."""
    data = adv_week[adv_week["advertiser_id"] == adv_id].copy()
    data = compute_metrics(data)
    for m in ["ivr", "cvr", "cpa", "cpv", "roas"]:
        if m in data.columns:
            data[m] = winsorize(data[m])
    data = data.merge(platform, on="week_start", how="inner")
    data = data.sort_values("week_start")

    # Add advertiser-specific covariate candidates
    data["metric_lag1"] = data[metric].shift(1)
    data["metric_lag2"] = data[metric].shift(2)
    if len(data) > 1:
        data["spend_change_pct"] = data["spend"].pct_change().fillna(0).clip(-1, 5)
    else:
        data["spend_change_pct"] = 0.0
    data = data.dropna(subset=["metric_lag1"]).set_index("week_start").sort_index()

    # Determine periods with ramp-up exclusion
    int_ts = pd.Timestamp(intervention_date)
    int_week = int_ts - pd.Timedelta(days=int_ts.weekday())
    post_start = int_week + pd.Timedelta(weeks=RAMP_UP_WEEKS)

    pre = data[data.index < int_week]
    post = data[data.index >= post_start]  # skip ramp-up weeks

    if len(pre) < MIN_PRE_WEEKS or len(post) < MIN_POST_WEEKS:
        return None
    if post["spend"].sum() < MIN_TOTAL_POST_SPEND:
        return None

    pre_period = [pre.index[0], pre.index[-1]]
    post_period = [post.index[0], post.index[-1]]

    # BIC covariate selection for this advertiser
    best_covs = select_covariates_bic(data, metric, COVARIATE_CANDIDATES, pre_period)

    # Build CI input: exclude ramp-up weeks from the data entirely
    ci_data = pd.concat([data.loc[:pre_period[1]], data.loc[post_period[0]:]])
    ci_data = ci_data[[metric] + best_covs].dropna(subset=[metric]).astype(float)
    ci_data[best_covs] = ci_data[best_covs].ffill().bfill()

    if len(ci_data) < 10:
        return None

    try:
        ci = CausalImpact(ci_data, pre_period, post_period)
        inf = ci.inferences[ci.inferences.index >= post_period[0]]
        predicted = inf["preds"].mean()
        abs_eff = inf["point_effects"].mean()
        return {
            "advertiser_id": adv_id,
            "pre_weeks": len(pre), "post_weeks": len(post),
            "actual_avg": ci.post_data.iloc[:, 0].mean(),
            "predicted_avg": predicted,
            "absolute_effect": abs_eff,
            "relative_effect": abs_eff / predicted if predicted != 0 else np.nan,
            "ci_lower": inf["point_effects_lower"].mean(),
            "ci_upper": inf["point_effects_upper"].mean(),
            "p_value": ci.p_value,
            "significant": ci.p_value < 0.05,
            "post_spend": float(post["spend"].sum()),
            "covariates_used": ", ".join(best_covs),
            "ci_object": ci,
        }
    except Exception as e:
        print(f"  Error: {e}")
        return None

In [ ]:
results = []
for _, row in adopters.iterrows():
    r = run_ci(row["advertiser_id"], row["intervention_date"], PRIMARY_METRIC, adv_week, platform)
    if r:
        r["company_name"] = row["company_name"]
        results.append(r)
        sig = "*" if r["significant"] else ""
        print(f"{row['company_name']:35s} ({row['advertiser_id']}): {r['relative_effect']:+.2%} "
              f"(p={r['p_value']:.4f}) {sig}  [{r['pre_weeks']}w pre, {r['post_weeks']}w post]"
              f"  covs: {r['covariates_used']}")
    else:
        print(f"{row['company_name']:35s} ({row['advertiser_id']}): skipped")

print(f"\n{len(results)} advertisers analyzed")
print(f"Ramp-up exclusion: first {RAMP_UP_WEEKS} weeks post-intervention excluded")

## 6. Results Summary

In [ ]:
def summarize(results, metric):
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "ci_object"} for r in results])
    df["effect_pct"] = df["relative_effect"].apply(lambda x: f"{x:+.2%}")
    df["p_fmt"] = df["p_value"].apply(lambda x: f"{x:.4f}")
    df["sig"] = df["significant"].apply(lambda x: "Yes" if x else "")

    display_cols = ["company_name", "advertiser_id", "pre_weeks", "post_weeks",
                    "actual_avg", "predicted_avg", "effect_pct", "p_fmt", "sig"]
    if "covariates_used" in df.columns:
        display_cols.append("covariates_used")

    display(df[display_cols]
           .style.set_caption(f"Causal Impact — {METRIC_DEFS[metric]['label']} (v5: BIC + ramp-up exclusion)"))

    d = METRIC_DEFS[metric]["direction"]
    n = len(df)
    n_sig = df["significant"].sum()
    n_pos = (df["relative_effect"] > 0).sum() if d == "higher" else (df["relative_effect"] < 0).sum()
    ts = df["post_spend"].sum()
    wt = (df["relative_effect"] * df["post_spend"]).sum() / ts if ts > 0 else 0

    print(f"\nAdvertisers analyzed:       {n}")
    print(f"Statistically significant:  {n_sig} ({n_sig/n:.0%})")
    print(f"Positive outcome:           {n_pos} ({n_pos/n:.0%})")
    print(f"Mean effect:                {df['relative_effect'].mean():+.2%}")
    print(f"Median effect:              {df['relative_effect'].median():+.2%}")
    print(f"Spend-weighted effect:      {wt:+.2%}")
    print(f"\nMethodology: BIC per-advertiser covariate selection, {RAMP_UP_WEEKS}-week ramp-up exclusion")
    return df

summary_ivr = summarize(results, PRIMARY_METRIC)

## 7. Per-Advertiser CausalImpact Plots

In [ ]:
for r in results:
    print(f"\n{'='*80}")
    print(f"{r['company_name']} ({r['advertiser_id']}) — {PRIMARY_METRIC.upper()}")
    print(f"Effect: {r['relative_effect']:+.2%} (p={r['p_value']:.4f})")
    print(f"{'='*80}")
    r["ci_object"].plot(figsize=(14, 10))
    plt.show()

## 8. Aggregate Summary Chart

In [ ]:
df_plot = summary_ivr.sort_values("relative_effect")
d = METRIC_DEFS[PRIMARY_METRIC]["direction"]
colors = ["#2ecc71" if (s and ((d=="higher" and e>0) or (d=="lower" and e<0)))
          else "#e74c3c" if s else "#95a5a6"
          for s, e in zip(df_plot["significant"], df_plot["relative_effect"])]

fig, ax = plt.subplots(figsize=(12, max(6, len(df_plot) * 0.6)))
bars = ax.barh(df_plot["company_name"], df_plot["relative_effect"] * 100, color=colors)
ax.axvline(x=0, color="black", linewidth=0.8)
for bar, pval in zip(bars, df_plot["p_value"]):
    x = bar.get_width()
    ax.text(x + (1 if x >= 0 else -1), bar.get_y() + bar.get_height() / 2,
            f"p={pval:.3f}", va="center", ha="left" if x >= 0 else "right", fontsize=9)
ax.set_xlabel("Relative Effect (%)")
ax.set_title(f"Media Plan Causal Impact — {PRIMARY_METRIC.upper()} (Recommended Plans Only)\n"
             "Green = significant positive | Red = significant negative | Gray = not significant")
plt.tight_layout()
plt.show()

## 9. Within-Advertiser Comparison

For advertisers with BOTH recommended and non-recommended campaigns running in the post-period (after 4-week ramp-up exclusion), compare their performance side-by-side.

**Important caveat:** Recommended campaigns are *new* campaigns (created with media plan), while non-recommended campaigns are typically *older* campaigns with established audience patterns. Even after excluding the 4-week ramp-up period (TI-780), remaining maturity differences may still confound this comparison.

In [ ]:
# Load recommended campaign group IDs
rec_cgs = set(client.query("""
SELECT mp.campaign_group_id
FROM `dw-main-silver.core.media_plan` mp
JOIN `dw-main-silver.core.media_plan_publishers` mpp ON mpp.media_plan_id = mp.media_plan_id
WHERE mp.media_plan_status_id = 3
GROUP BY 1
HAVING LOGICAL_AND(mpp.badge_state = 'RECOMMENDED')
""").to_dataframe()["campaign_group_id"].tolist())

comparison = []
for _, row in adopters.iterrows():
    adv_id = row["advertiser_id"]
    int_ts = pd.Timestamp(row["intervention_date"])
    int_week = int_ts - pd.Timedelta(days=int_ts.weekday())
    post_start = int_week + pd.Timedelta(weeks=RAMP_UP_WEEKS)  # exclude ramp-up

    post = weekly_kpis[(weekly_kpis["advertiser_id"] == adv_id) & (weekly_kpis["week_start"] >= post_start)]
    rec = post[post["campaign_group_id"].isin(rec_cgs)]
    non = post[~post["campaign_group_id"].isin(rec_cgs)]
    if rec.empty: continue
    r_agg = rec[["impressions","spend","vv","conversions","order_value"]].sum()
    n_agg = non[["impressions","spend","vv","conversions","order_value"]].sum() if not non.empty else pd.Series({"impressions":0,"spend":0,"vv":0,"conversions":0,"order_value":0})
    comparison.append({
        "company_name": row["company_name"], "advertiser_id": adv_id,
        "has_non_rec": not non.empty,
        "rec_ivr": r_agg["vv"]/r_agg["impressions"] if r_agg["impressions"]>0 else np.nan,
        "non_rec_ivr": n_agg["vv"]/n_agg["impressions"] if n_agg["impressions"]>0 else np.nan,
        "rec_cvr": r_agg["conversions"]/r_agg["vv"] if r_agg["vv"]>0 else np.nan,
        "non_rec_cvr": n_agg["conversions"]/n_agg["vv"] if n_agg["vv"]>0 else np.nan,
        "rec_spend": r_agg["spend"], "non_rec_spend": n_agg["spend"],
    })

comp_df = pd.DataFrame(comparison)
display(comp_df.style.set_caption(
    f"Within-Advertiser: Recommended vs Non-Recommended (Post-Period, {RAMP_UP_WEEKS}-week ramp-up excluded)"
).format(precision=4))

both = comp_df[comp_df["has_non_rec"]]
if not both.empty:
    print(f"\nAdvertisers with both types: {len(both)}")
    print(f"Avg IVR diff (rec - non_rec): {(both['rec_ivr'] - both['non_rec_ivr']).mean():+.4f}")

## 10. All Metrics (Optional)

In [ ]:
all_summaries = {}
for metric in METRIC_DEFS:
    print(f"\n{'='*60}\n{metric.upper()} ({METRIC_DEFS[metric]['label']})\n{'='*60}")
    mr = []
    for _, row in adopters.iterrows():
        r = run_ci(row["advertiser_id"], row["intervention_date"], metric, adv_week, platform)
        if r:
            r["company_name"] = row["company_name"]
            mr.append(r)
    if mr:
        all_summaries[metric] = summarize(mr, metric)

## 11. Cross-Metric Summary

In [ ]:
cross = []
for metric, df in all_summaries.items():
    info = METRIC_DEFS[metric]
    n = len(df); n_sig = df["significant"].sum()
    n_pos = (df["relative_effect"] > 0).sum() if info["direction"] == "higher" else (df["relative_effect"] < 0).sum()
    ts = df["post_spend"].sum()
    wt = (df["relative_effect"] * df["post_spend"]).sum() / ts if ts > 0 else 0
    cross.append({"Metric": metric.upper(), "Label": info["label"], "N": n,
                  "Significant": f"{n_sig}/{n}", "Positive": f"{n_pos}/{n}",
                  "Mean": f"{df['relative_effect'].mean():+.2%}",
                  "Median": f"{df['relative_effect'].median():+.2%}",
                  "Spend-Weighted": f"{wt:+.2%}"})
display(pd.DataFrame(cross).style.set_caption("Cross-Metric Summary"))

## 12. Panel Data Model (Two-Way Fixed Effects)

Complementary aggregate analysis: pool all adopter advertisers into a single regression with advertiser fixed effects, time trend, and a treatment indicator (post-intervention AND past ramp-up). This produces one aggregate treatment effect estimate with proper standard errors.

**Why both approaches:**
- **Per-advertiser CausalImpact** shows WHO benefits and by how much (heterogeneous effects)
- **Panel model** gives ONE population-level estimate with more statistical power (pools data across units)

In [ ]:
def run_panel_model(adv_week, adopters, platform, metric):
    """Panel regression with advertiser fixed effects, time trend, and treatment indicator."""
    rows = []
    for _, adv_row in adopters.iterrows():
        adv_id = adv_row["advertiser_id"]
        intervention = pd.Timestamp(adv_row["intervention_date"])
        intervention_week = intervention - pd.Timedelta(days=intervention.weekday())
        post_start = intervention_week + pd.Timedelta(weeks=RAMP_UP_WEEKS)

        data = adv_week[adv_week["advertiser_id"] == adv_id].copy()
        data = compute_metrics(data)
        data = data.merge(platform, on="week_start", how="inner")
        data = data.sort_values("week_start")
        if len(data) < 3:
            continue
        data["spend_change_pct"] = data["spend"].pct_change().fillna(0).clip(-1, 5)
        data["metric_lag1"] = data[metric].shift(1)
        data = data.dropna(subset=["metric_lag1"])
        if len(data) < 20:
            continue

        data["treated"] = (data["week_start"] >= post_start).astype(float)
        data["ramp_up"] = ((data["week_start"] >= intervention_week) & (data["week_start"] < post_start)).astype(float)
        data["adv_id"] = adv_id
        data["time_idx"] = (data["week_start"] - data["week_start"].min()).dt.days / 7
        rows.append(data)

    if not rows:
        return None

    panel = pd.concat(rows, ignore_index=True)
    adv_dummies = pd.get_dummies(panel["adv_id"], prefix="adv", drop_first=True, dtype=float)

    X_cols = ["treated", "ramp_up", "time_idx", "holiday", "spend_change_pct", "metric_lag1"]
    X = pd.concat([panel[X_cols], adv_dummies], axis=1)
    X = sm.add_constant(X)
    y = panel[metric].astype(float)

    mask = X.notna().all(axis=1) & y.notna()
    X, y = X[mask], y[mask]

    if len(y) < 50:
        return None

    model = sm.OLS(y, X.astype(float)).fit(cov_type="HAC", cov_kwds={"maxlags": 4})

    coeff = model.params["treated"]
    rel_effect = coeff / y.mean() if y.mean() != 0 else np.nan

    print(f"Treatment effect:    {coeff:.6f}")
    print(f"Relative effect:     {rel_effect:+.2%}")
    print(f"p-value:             {model.pvalues['treated']:.4f}")
    print(f"95% CI:              [{model.conf_int().loc['treated'][0]:.6f}, {model.conf_int().loc['treated'][1]:.6f}]")
    print(f"Significant:         {'Yes' if model.pvalues['treated'] < 0.05 else 'No'}")
    print(f"Ramp-up effect:      {model.params.get('ramp_up', np.nan):.6f} (p={model.pvalues.get('ramp_up', np.nan):.4f})")
    print(f"N observations:      {len(y)}")
    print(f"N advertisers:       {panel['adv_id'].nunique()}")
    print(f"R² (adj):            {model.rsquared_adj:.4f}")
    return model

print("Running panel data model for IVR...")
panel_model = run_panel_model(adv_week, adopters, platform, PRIMARY_METRIC)

---

## Appendix A: How CausalImpact Works

### The Core Idea

We can't directly observe what *would have happened* to an advertiser's performance if they hadn't adopted media plan (the **counterfactual**). CausalImpact solves this by building a statistical model that learns the advertiser's behavior patterns from the pre-period, then uses that model to predict what *should have happened* in the post-period.

The difference between the **actual** post-period performance and the **predicted** (counterfactual) performance is the estimated causal effect.

### Step-by-Step Process (v5)

```
PRE-PERIOD (52 weeks before adoption):
┌─────────────────────────────────────────────────────────────┐
│ 1. VIF: Remove collinear covariates (VIF > 10)             │
│ 2. BIC: Test all subsets, select lowest-BIC per advertiser  │
│ 3. Model learns: IVR = f(BIC-selected covariates)          │
└─────────────────────────────────────────────────────────────┘
                              │
                    RAMP-UP (4 weeks) ← EXCLUDED
                              │
                              ▼
POST-PERIOD (after ramp-up):
┌─────────────────────────────────────────────────────────────┐
│ Model predicts counterfactual using learned relationships   │
│ Causal Effect = Actual - Predicted                          │
│ Relative Effect = (Actual - Predicted) / Predicted          │
└─────────────────────────────────────────────────────────────┘
```

### The Statistical Model (Bayesian Structural Time Series)

Under the hood, CausalImpact uses a **Bayesian structural time series** model with three components:

1. **Local linear trend:** Captures the advertiser's own trajectory (is IVR trending up or down over time?)
2. **Regression on covariates:** Learns how the advertiser's metric relates to external factors — selected via BIC per advertiser, NOT hand-picked
3. **Bayesian inference:** Instead of a single prediction, the model generates a **probability distribution** of possible counterfactuals, giving us uncertainty bounds.

### How P-Values Are Calculated

CausalImpact computes p-values using a **Bayesian tail-area probability:**

1. The model generates thousands of simulated counterfactual paths from the posterior distribution
2. For each simulation, it calculates the cumulative effect (sum of pointwise differences)
3. The p-value = the fraction of simulated effects that are more extreme than zero in the same direction as the observed effect
4. If p = 0.02, it means only 2% of the simulated counterfactuals showed an effect as large as what we observed — strong evidence that the intervention had a real impact

**Note:** These are *one-sided* p-values. p < 0.05 means there's < 5% chance the observed effect occurred by chance.

### Why BIC Per-Advertiser (v5 improvement)

In v2, we hand-picked 7 covariates for all advertisers. This produced:
- 86% placebo false positive rate (most "significant" results were noise)
- Distorted effects (FICO went from -14.8% to -4.0% after fixing covariates)

In v5, BIC selects the best 2-4 covariates per advertiser from 14 candidates:
- **Result:** Placebo FPR dropped to 24%, effects stabilized
- **Key finding:** `spend_change_pct` was selected for ALL advertisers. Platform-wide metrics were mostly eliminated — advertiser-specific dynamics dominate.

### Why Ramp-Up Exclusion (v5 improvement)

New campaigns underperform during their first 4 weeks (TI-780, N=6,917):
- Week 0: 38% of steady-state IVR
- Week 4: 89% of steady state (ramp-up complete)
- Driven by bidder learning, frequency buildup, delivery footprint exploration

Including ramp-up weeks in the post-period attributes this natural startup noise to the media plan intervention, inflating negative effects for recently-adopted advertisers.

### v5 Covariates

| Covariate | What It Controls For | Selected By |
|---|---|---|
| `spend_change_pct` | Week-over-week budget changes (strongest predictor — selected for ALL advertisers) | BIC |
| `metric_lag1` / `metric_lag2` | Advertiser's own momentum/autocorrelation | BIC |
| `holiday` | Known seasonal spikes (Black Friday, Christmas, Super Bowl) | BIC |
| `platform_ivr` | Market-wide engagement trends | VIF usually eliminates; BIC rarely selects |
| `platform_spend` | Industry spending patterns | VIF usually eliminates |
| `platform_impressions` | Supply-side changes | VIF usually eliminates |
| `platform_active_advertisers` | Competition changes | VIF usually eliminates |
| `platform_vcr` | Video completion rate | VIF usually eliminates |

### Key Assumptions

1. **Parallel trends:** Without the intervention, the advertiser would have continued following the pattern predicted by covariates
2. **No spillover:** Media plan adoption doesn't affect non-adopter performance (our covariates)
3. **Stable relationships:** The covariate relationships learned in pre-period remain valid in post-period

### Limitations

1. **Small N:** Only ~8 analyzable advertisers. Individual results are noisy.
2. **Selection bias:** Advertisers who chose to adopt may be systematically different (more engaged, growing faster).
3. **Confounders:** Other changes (new creatives, budget shifts, audience changes) happening simultaneously.
4. **Campaign maturity:** Even with 4-week ramp-up exclusion, newer campaigns may still underperform mature ones.
5. **Placebo FPR of 24%:** Model is good but not perfect — interpret p-values near 0.05 with caution.